# Module 4 — frozen modelling population and chronological split

This notebook section fixes the population and temporal validation design before model comparison.

- Source: `data/interim/sample_data_intw_cleaned.csv`
- Date column: `pdate`
- Original outcome: `label` (`1` = repaid within five days; `0` = delinquent)
- Modelling target: `delinquent_5d` (`1` = delinquent)
- Development period: through 13 July 2016
- Final untouched holdout: 14–23 July 2016
- Post-23 July records: exploratory robustness/distribution checks only
- `msisdn`: validation checks only, not a predictor


In [ ]:
from pathlib import Path
import json
import pandas as pd

DATA_PATH = Path('data/interim/sample_data_intw_cleaned.csv')
SUMMARY_PATH = Path('reports/tables/module4_split_summary.json')
HOLDOUT_START = pd.Timestamp('2016-07-14')
MODELLING_CUTOFF = pd.Timestamp('2016-07-23')

df = pd.read_csv(DATA_PATH)
required = {'pdate', 'label', 'msisdn'}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

df['pdate'] = pd.to_datetime(df['pdate'], dayfirst=True, errors='raise')
df['delinquent_5d'] = 1 - df['label'].astype(int)

print(f'Cleaned source rows: {len(df):,}')
print(f"Date range: {df['pdate'].min().date()} to {df['pdate'].max().date()}")
print('Original label values:', sorted(df['label'].dropna().unique().tolist()))
print('Delinquency target values:', sorted(df['delinquent_5d'].dropna().unique().tolist()))


In [ ]:
development_customers = set(development['msisdn)].dropna())
holdout_customer_seen = final_holdout['msisdn'].isin(development_customers)
customer_check = pd.Series({
    'holdout_rows': len(final_holdout),
    'holdout_rows_seen_customer': int(holdout_customer_seen.sum()),
    'holdout_rows_unseen_customer': int((~holdout_customer_seen).sum()),
    'share_unseen_customer': float((~holdout_customer_seen).mean()),
}, name='value')
customer_check


## Interpretation to carry into the report

The labelled data cover only a short historical period. A chronological holdout is therefore used to test short-term temporal generalisation rather than a random split that mixes earlier and later observations. Data through 13 July are available for development and time-aware cross-validation. The period from 14–23 July is reserved as the untouched final holdout.

The post-23 July observations are kept separate from supervised performance evaluation. They can later be used for exploratory checks of feature distributions, score behaviour and robustness, but their observed outcomes are not used to calculate model performance.

Calendar-position variables may be explored as candidate features, but the short history does not support treating an apparent day-of-month pattern as a stable seasonal effect.
